# Match up everyone

In [60]:
# imports
from importlib import reload
#import earthaccess

import os
import numpy as np

import pandas
from matplotlib import pyplot as plt

from shapely.geometry import Polygon, Point
from shapely.vectorized import contains

from ocpy.utils import io as ocpy_io

from remote_sensing.plotting import globe
from remote_sensing.download import earthaccess as rs_ea

import grab_pace_granules
import argo

# Load up BGC

In [28]:
reload(argo)
argo_pace = argo.load_argo()

In [29]:
argo_pace.head()

,cruise,filename,profile,lat,lon,time,solar_angle
261,1902304,1902304QC.nc,158,56.325,-18.987,2024-04-10 21:11:00.000001024+00:00,-8.083862
262,1902304,1902304QC.nc,159,56.290,-18.758,2024-04-20 21:24:00.000004608+00:00,-6.925605
263,1902304,1902304QC.nc,160,56.509,-18.912,2024-04-30 21:30:00.000003584+00:00,-4.744463
264,1902304,1902304QC.nc,161,57.344,-19.839,2024-05-10 20:40:00.000001024+00:00,4.443112
265,1902304,1902304QC.nc,163,57.463,-19.798,2024-05-30 20:25:00.000004608+00:00,9.799566


# Load PACE granules

In [5]:
reload(grab_pace_granules)
granules, pace = grab_pace_granules.load_from_json('PACE_50clouds.json')

In [6]:
pace.head()

,id,polygon,time,CC,url
0,PACE_OCI_L2_AOP_PACE_OCI.20240305T045359.L2.OC...,"POLYGON ((129.99634 19.95012, 105.26324 14.805...",2024-03-05 04:56:28.500000+00:00,49.5,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...
1,PACE_OCI_L2_AOP_PACE_OCI.20240305T081040.L2.OC...,"POLYGON ((80.81132 19.9969, 56.07363 14.85543,...",2024-03-05 08:13:10+00:00,48.2,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...
2,PACE_OCI_L2_AOP_PACE_OCI.20240305T112721.L2.OC...,"POLYGON ((31.63485 20.04028, 6.9071 14.89843, ...",2024-03-05 11:29:50.500000+00:00,42.7,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...
3,PACE_OCI_L2_AOP_PACE_OCI.20240305T143902.L2.OC...,"POLYGON ((-14.32906 2.23116, -38.0179 -2.80614...",2024-03-05 14:41:31.500000+00:00,46.1,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...
4,PACE_OCI_L2_AOP_PACE_OCI.20240305T162722.L2.OC...,"POLYGON ((-44.22893 37.98507, -73.20055 32.449...",2024-03-05 16:29:51.500000+00:00,48.9,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...


# In a Polygon?

In [16]:
argo0 = argo_pace.iloc[0]
point0 = Point(argo0.lon, argo0.lat)
argo0

cruise                               1902304
filename                        1902304QC.nc
profile                                  158
lat                                   56.325
lon                                  -18.987
time           2024-04-10 21:11:00.000001024
solar_angle                        -8.083862
Name: 261, dtype: object

In [18]:
in_poly = [poly.contains(point0) for poly in pace.polygon.values]
in_poly = np.array(in_poly)

In [31]:
pace.time

0      2024-03-05 04:56:28.500000+00:00
1             2024-03-05 08:13:10+00:00
2      2024-03-05 11:29:50.500000+00:00
3      2024-03-05 14:41:31.500000+00:00
4      2024-03-05 16:29:51.500000+00:00
                     ...               
4649   2025-05-01 16:39:18.500000+00:00
4650   2025-05-01 16:54:18.500000+00:00
4651   2025-05-01 20:01:01.500000+00:00
4652   2025-05-01 20:16:01.500000+00:00
4653   2025-05-01 21:39:22.500000+00:00
Name: time, Length: 4654, dtype: datetime64[ns, UTC]

In [46]:
dt = pace.time - pandas.Timestamp(argo0.time, tz='UTC')
good_dt = np.abs(dt) < pandas.Timedelta('1 day')

In [47]:
np.any(in_poly & good_dt)

np.False_

## Loop over em

In [84]:
# too slow
'''
match_up = []
for ss in range(len(argo_pace)):
    iargo = argo_pace.iloc[ss]
    # In granule?
    point = Point(iargo.lon, iargo.lat)
    in_poly = [poly.contains(point) for poly in pace.polygon.values]
    in_poly = np.array(in_poly)
    # In time window?
    dt = pace.time - pandas.Timestamp(iargo.time)#, tz='UTC')
    good_dt = np.abs(dt) < pandas.Timedelta('1 day')
    # Ok for both?
    match_up.append(np.any(in_poly & good_dt))
'''

"\nmatch_up = []\nfor ss in range(len(argo_pace)):\n    iargo = argo_pace.iloc[ss]\n    # In granule?\n    point = Point(iargo.lon, iargo.lat)\n    in_poly = [poly.contains(point) for poly in pace.polygon.values]\n    in_poly = np.array(in_poly)\n    # In time window?\n    dt = pace.time - pandas.Timestamp(iargo.time)#, tz='UTC')\n    good_dt = np.abs(dt) < pandas.Timedelta('1 day')\n    # Ok for both?\n    match_up.append(np.any(in_poly & good_dt))\n"

----

In [55]:
points = [Point(iargo.lon, iargo.lat) for _, iargo in argo_pace.iterrows()]

In [58]:
inside = [pace.polygon.values[0].contains(point) for point in points]

## Inside?

In [69]:
all_inside = []
for ss in range(len(pace)):
    inside = contains(pace.polygon.values[ss], argo_pace.lon, argo_pace.lat)
    all_inside.append(np.array(inside))
all_inside = np.array(all_inside)

## Ok time?

In [76]:
ok_times = []
for ss in range(len(pace)):
    dt = pandas.Timestamp(pace.iloc[ss].time) - argo_pace.time
    good_dt = np.abs(dt) < pandas.Timedelta('1 day')
    ok_times.append(np.array(good_dt))
ok_times = np.array(ok_times)

In [77]:
ok_times.shape

(4654, 5954)

In [79]:
match = all_inside & ok_times

In [82]:
good_argo = np.any(match, axis=0)

In [83]:
np.sum(good_argo)

np.int64(1255)